# Final Engineering Assessment: Problem → Optimisation → Scalability

This notebook converts the completed electrical, coupled, thermal, geometry, mass and manufacturability results into one traceable engineering decision.

**Final scope:** heat-sink material and geometry are compared while **TGP5000 (1.5 mm, 5 W/mK) is fixed**. TIM optimisation and aluminium/copper hybrid construction are outside the final scope.

**Design requirement:** maintain **Tj ≤ 125°C at 15 W imposed thermal dissipation and 25°C ambient**. The 175°C datasheet value is treated as an absolute limit.

**Optimisation objective:** after material screening, select the **minimum-mass tested aluminium geometry** that satisfies the 125°C thermal constraint.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

master = pd.read_csv("master_results.csv")
mass_data = pd.read_csv("../data/heatsink_mass_comparison.csv")

master.head()

,Case_ID,Scenario_Type,Load_Point,Load_Current_A,Heat_Load_W,Material,TIM_Config,Geometry,Ambient_Temperature_C,Junction_Temperature_C,MOSFET_Loss_W,Efficiency_pct,Margin_to_125C_C,Margin_to_175C_C,Mass_kg,Estimated_Cost,Cost_Unit,Manufacturability,Status,Notes
0,C01,Electrical,Low load,5.0,0.534,Aluminium,TGP5000,Baseline,25,27.8159,0.5374,99.1122,97.1841,147.1841,0.3510,0.829293,GBP raw material,High - straight-fin aluminium geometry is extr...,Converged,Coupled electrically derived case
1,C02,Electrical,Baseline,10.0,1.505,Aluminium,TGP5000,Baseline,25,33.1078,1.5475,98.7268,91.8922,141.8922,0.3510,0.829293,GBP raw material,High - straight-fin aluminium geometry is extr...,Converged,Coupled baseline electrical case
2,C03,Electrical,High load,20.0,4.760,Aluminium,TGP5000,Baseline,25,53.1405,5.3713,97.8110,71.8595,121.8595,0.3510,0.829293,GBP raw material,High - straight-fin aluminium geometry is extr...,Converged,Coupled high-current electrical case
3,C04,Electrical,Low load,5.0,0.534,Copper,TGP5000,Baseline,25,27.7450,0.5373,99.1124,97.2550,147.2550,1.1648,11.790781,GBP raw material,Moderate - thermally capable but high mass and...,Converged,Coupled material-comparison case
4,C05,Electrical,Baseline,10.0,1.505,Copper,TGP5000,Baseline,25,32.8997,1.5464,98.7277,92.1003,142.1003,1.1648,11.790781,GBP raw material,Moderate - thermally capable but high mass and...,Converged,Coupled material-comparison case


## 1. Engineering Problem: Is Dedicated Cooling Necessary?

Before optimising the heat sink, the project must establish that cooling is actually required. The 10 A no-heat-sink coupled reference is compared directly with the cooled baseline designs.


In [2]:
cooling_need = master[master["Case_ID"].isin(["C11", "C02", "C05"])][
    ["Case_ID", "Material", "Junction_Temperature_C", "Status"]
].copy()
cooling_need["Configuration"] = ["No heat sink", "80 mm aluminium", "80 mm copper"]
cooling_need[["Configuration", "Junction_Temperature_C", "Status"]].round(3)

,Configuration,Junction_Temperature_C,Status
1,No heat sink,33.108,Converged
4,80 mm aluminium,32.900,Converged
10,80 mm copper,183.844,Exceeded 175C limit


The no-heat-sink case reaches approximately **183.8°C** before a safe converged solution is obtained, exceeding the **175°C absolute device limit**. The baseline aluminium heat sink reduces the coupled 10 A junction temperature to approximately **33.1°C**.

The dominant first design decision is therefore clear: **dedicated cooling is required**. The optimisation question is then how much passive cooling hardware is justified.


## 2. Electrical Operating Envelope and Electro-Thermal Feedback

The 5 A, 10 A and 20 A cases define the electrically derived operating envelope. The imposed 10 W and 15 W cases are treated separately and are not mapped to converter current.


In [3]:
electrical_al = master[(master["Scenario_Type"] == "Electrical") & (master["Material"] == "Aluminium")].sort_values("Load_Current_A")
one_pass = {5: 0.534, 10: 1.505, 20: 4.760}
summary = electrical_al[["Load_Current_A","MOSFET_Loss_W","Junction_Temperature_C","Margin_to_125C_C"]].copy()
summary["One_pass_loss_W"] = summary["Load_Current_A"].map(one_pass)
summary["Feedback_increase_pct"] = 100*(summary["MOSFET_Loss_W"]-summary["One_pass_loss_W"])/summary["One_pass_loss_W"]
summary[["Load_Current_A","One_pass_loss_W","MOSFET_Loss_W","Feedback_increase_pct","Junction_Temperature_C","Margin_to_125C_C"]].round(3)

,Load_Current_A,One_pass_loss_W,MOSFET_Loss_W,Feedback_increase_pct,Junction_Temperature_C,Margin_to_125C_C
0,5.0,0.534,0.537,0.637,27.816,97.184
1,10.0,1.505,1.548,2.824,33.108,91.892
2,20.0,4.760,5.371,12.842,53.140,71.860


MOSFET loss grows strongly with current, while the temperature-dependent increase in `RDS(on)` makes the coupled correction more important at high current. At 20 A the coupled loss is approximately **12.8%** above the one-pass 25°C estimate.

This is the electrical side of scalability: increasing electrical demand does not only increase instantaneous loss; the resulting temperature feeds back into conduction loss.


## 3. Coupling Robustness to Initial Temperature

The 20 A aluminium case is repeated from three initial junction-temperature guesses using the same coupled model.


In [4]:
Vin = 24.0
fs = 50e3
D = 0.5
tr = 60e-9
tf = 45e-9
Rds_25 = 0.0175
temperature_points = np.array([25, 50, 75, 100, 125, 150, 175])
rds_factors = np.array([1.00, 1.15, 1.35, 1.58, 1.82, 2.10, 2.39])
thermal_power = np.array([0.534, 1.505, 4.760, 10.0, 15.0])
thermal_temperature = np.array([27.798, 32.885, 49.938, 77.390, 103.595])

def rds_on(Tj):
    return Rds_25*np.interp(Tj, temperature_points, rds_factors)

def loss(current, Tj):
    return current**2*rds_on(Tj)*D + 0.5*Vin*current*(tr+tf)*fs

def thermal_T(power):
    return np.interp(power, thermal_power, thermal_temperature)

def coupled_from_start(initial_Tj):
    Tj = initial_Tj
    for iteration in range(1, 21):
        p = loss(20, Tj)
        new_Tj = thermal_T(p)
        if abs(new_Tj-Tj) < 0.1:
            return new_Tj, p, iteration
        Tj = new_Tj
    return Tj, p, iteration

robustness=[]
for start in [25.0, 75.0, 125.0]:
    Tj,p,it=coupled_from_start(start)
    robustness.append({"Initial Tj (°C)":start,"Final Tj (°C)":Tj,"Final loss (W)":p,"Iterations":it})
robustness=pd.DataFrame(robustness)
robustness.round(4)

,Initial Tj (°C),Final Tj (°C),Final loss (W),Iterations
0,25.0,53.1405,5.3713,4
1,75.0,53.1607,5.3751,4
2,125.0,53.1561,5.3743,5


All three runs converge to approximately **53.14-53.16°C** and **5.37 W**, with less than about **0.02°C** spread in final temperature. This supports numerical robustness of the fixed-point coupling result to the starting guess.


## 4. Material Screening Before Geometry Optimisation

Material is screened first using equal geometry so that the later geometry optimisation is not confounded by changing both material and shape at once.


In [5]:
material_screen = master[master["Case_ID"].isin(["C09","C10"])][
    ["Material","Junction_Temperature_C","Mass_kg","Estimated_Cost","Cost_Unit"]
].copy()
material_screen["Temperature_benefit_vs_Al_C"] = material_screen["Junction_Temperature_C"].iloc[0] - material_screen["Junction_Temperature_C"]
material_screen.round(3)

,Material,Junction_Temperature_C,Mass_kg,Estimated_Cost,Cost_Unit,Temperature_benefit_vs_Al_C
8,Aluminium,103.595,0.351,0.829,GBP raw material,0.000
9,Copper,101.624,1.165,11.791,GBP raw material,1.971


At 15 W, copper lowers junction temperature by only approximately **1.97°C**, while equal-geometry copper is approximately **3.32× heavier** and much more expensive on a raw-material basis.

**Material screening result: aluminium is selected as the default system-level material.** Copper remains a technically viable alternative only when the small additional thermal margin is worth its mass/cost penalty.


## 5. Discrete Constrained Geometry Optimisation

The geometry task is formulated as:

**Objective:** minimise aluminium heat-sink mass / raw-material use.  
**Constraint:** `Tj ≤ 125°C` at 15 W and 25°C ambient.  
**Candidates:** 80 mm / 8 fins, 60 mm / 6 fins, 40 mm / 4 fins.

This is a **discrete candidate optimisation**, not a continuous fin/topology optimisation.


In [6]:
# 15 W aluminium cases: original C09, 60 mm C18, 40 mm C16
opt = master[master["Case_ID"].isin(["C09","C18","C16"])][
    ["Case_ID","Geometry","Junction_Temperature_C","Margin_to_125C_C","Mass_kg","Estimated_Cost","Cost_Unit"]
].copy()
name_map={"C09":"Original 80 mm / 8 fins","C18":"Geometry 1 - 60 mm / 6 fins","C16":"Geometry 2 - 40 mm / 4 fins"}
opt["Candidate"] = opt["Case_ID"].map(name_map)
opt["Mass_reduction_vs_original_pct"] = 100*(0.351-opt["Mass_kg"])/0.351
opt["Feasible"] = opt["Junction_Temperature_C"] <= 125
opt = opt.sort_values("Mass_kg", ascending=False)
opt[["Candidate","Junction_Temperature_C","Margin_to_125C_C","Mass_kg","Mass_reduction_vs_original_pct","Estimated_Cost","Feasible"]].round(3)

,Candidate,Junction_Temperature_C,Margin_to_125C_C,Mass_kg,Mass_reduction_vs_original_pct,Estimated_Cost,Feasible
8,Original 80 mm / 8 fins,103.595,21.405,0.351,0.0,0.829,True
17,Geometry 1 - 60 mm / 6 fins,111.831,13.169,0.263,25.0,0.622,True
15,Geometry 2 - 40 mm / 4 fins,129.400,-4.400,0.176,50.0,0.415,False


**Optimisation result:** Geometry 1 (60 mm / 6 fins) is the **minimum-mass tested aluminium design that satisfies the 125°C requirement at 15 W**.

- 80 mm: feasible, but heavier than necessary for the stated requirement.
- 60 mm: feasible and 25% lighter than the original.
- 40 mm: lightest, but infeasible because Tj reaches 129.4°C.

At the controlled 1.505 W fixed heat load, the original / 60 mm / 40 mm aluminium temperatures are **32.885°C / 33.713°C / 35.475°C**. The 60 mm design therefore removes 25% of heat-sink mass for only about **0.83°C** baseline temperature penalty.


## 6. Cooling-System Scalability

Under the approximately linear steady-state assumptions used in the thermal model, a geometry with higher effective thermal resistance accumulates a larger **absolute temperature penalty** as heat load rises. The 15 W results can therefore be used to estimate where each aluminium design reaches the 125°C design target.


In [7]:
capacity = opt[["Candidate","Junction_Temperature_C"]].copy()
capacity["Effective_Rth_C_per_W"] = (capacity["Junction_Temperature_C"] - 25.0)/15.0
capacity["Estimated_heat_load_at_125C_W"] = (125.0-25.0)/capacity["Effective_Rth_C_per_W"]
capacity[["Candidate","Effective_Rth_C_per_W","Estimated_heat_load_at_125C_W"]].round(3)

,Candidate,Effective_Rth_C_per_W,Estimated_heat_load_at_125C_W
8,Original 80 mm / 8 fins,5.240,19.085
17,Geometry 1 - 60 mm / 6 fins,5.789,17.275
15,Geometry 2 - 40 mm / 4 fins,6.960,14.368


The approximate 125°C heat-load thresholds are **19.1 W (80 mm)**, **17.3 W (60 mm)** and **14.4 W (40 mm)**.

These values are **estimated thermal-capacity thresholds**, not validated safety limits. They provide the scalability story: as required heat dissipation increases, the 40 mm design loses compliance first, the 60 mm design provides an intermediate lightweight range, and the 80 mm design provides the greatest thermal headroom.


## 7. Natural-Convection Uncertainty

Passive natural convection is uncertain. A first-order analytical sensitivity is therefore used to test whether the geometry recommendation depends strongly on the assumed convection coefficient. The calculation uses the fixed TIM, aluminium properties and geometry-derived exposed areas.

This is an **uncertainty study**, not a replacement for the ANSYS simulations.


In [8]:
Ta = 25.0
P = 15.0
R_jc = 1.5
t_tim = 1.5e-3
k_tim = 5.0
A_contact = 15.8e-3 * 10.0e-3
t_base = 5e-3
k_al = 170.0

R_tim = t_tim/(k_tim*A_contact)
R_base = t_base/(k_al*A_contact)

# Geometry-derived first-order exposed areas from the documented rectangular base/fins.
areas = {
    "Original 80 mm": 0.0537,
    "Geometry 1 - 60 mm": 0.0404,
    "Geometry 2 - 40 mm": 0.0271,
}
rows=[]
for design,A in areas.items():
    for h in [5.0,10.0,15.0]:
        R_total = R_jc + R_tim + R_base + 1/(h*A)
        Tj = Ta + P*R_total
        rows.append({"Design":design,"h (W/m²K)":h,"Analytical Tj at 15 W (°C)":Tj,"Meets 125°C":Tj<=125})
conv_sensitivity=pd.DataFrame(rows)
conv_sensitivity.pivot(index="Design",columns="h (W/m²K)",values="Analytical Tj at 15 W (°C)").round(1)

h (W/m²K),5.0,10.0,15.0
Design,,,
Geometry 1 - 60 mm,153.0,115.9,103.5
Geometry 2 - 40 mm,189.5,134.1,115.7
Original 80 mm,134.6,106.7,97.4


At the baseline assumption **h = 10 W/m²K**, the analytical sensitivity preserves the key decision: the 60 mm geometry remains below 125°C while the 40 mm geometry does not.

At **h = 5 W/m²K**, none of the tested passive designs meets the 125°C / 15 W requirement. This is an important limitation rather than a failed result: the recommendation is conditional on the natural-convection environment, and poorer convection would require a larger heat sink, lower dissipation or a different cooling strategy.


## 8. Manufacturability and Industrial Scalability

The straight-fin aluminium concept is compatible with extrusion, but the tested geometries are **not one identical profile cut to different lengths**. The 80 / 60 / 40 mm dimension is across the fin array and the fin count changes 8 / 6 / 4, so the cross-section changes.

A defensible industrial interpretation is:

- **low volume / prototype:** machine or trim suitable standard stock where practical, avoiding custom-die cost;
- **high volume:** justify a dedicated extrusion die/profile for the selected final geometry;
- **system integration:** the 60 mm aluminium design reduces material use, mounting load and handling burden relative to the 80 mm baseline;
- **copper:** reserved for cases where the small extra thermal margin justifies much higher equal-geometry mass and material cost.


## 9. Estimated Current Thresholds (Extrapolative)

The original notebook labelled these results "safe current limits". That wording is too strong because the reduced geometries have only 1.505 W and 15 W ANSYS points and currents near the thermal limit require interpolation/extrapolation beyond the directly simulated range.

The calculation is retained as a **comparative estimate of current capability**, not a validated safety rating.


In [9]:
designs = {
    "Baseline 80 mm": [(0.0,25.0),(0.534,27.798),(1.505,32.885),(4.760,49.938),(10.0,77.390),(15.0,103.595)],
    "Geometry 1 - 60 mm": [(0.0,25.0),(1.505,33.7125),(15.0,111.831)],
    "Geometry 2 - 40 mm": [(0.0,25.0),(1.505,35.4745),(15.0,129.400)],
}

def thermal_temperature_extrap(power, points):
    powers=np.array([p for p,T in points]); temps=np.array([T for p,T in points])
    if power <= powers[-1]:
        return np.interp(power,powers,temps)
    slope=(temps[-1]-temps[-2])/(powers[-1]-powers[-2])
    return temps[-1]+slope*(power-powers[-1])

def coupled_temperature(current, points):
    Tj=25.0
    for _ in range(100):
        p=loss(current,Tj)
        new=thermal_temperature_extrap(p,points)
        if abs(new-Tj)<0.1:
            return new
        Tj=new
    return Tj

def find_threshold(target, points):
    lo,hi=0.0,50.0
    for _ in range(100):
        mid=(lo+hi)/2
        if coupled_temperature(mid,points)<target: lo=mid
        else: hi=mid
    return (lo+hi)/2

threshold_rows=[]
for design,points in designs.items():
    threshold_rows.append({
        "Design":design,
        "Estimated current at 125°C (A)":find_threshold(125,points),
        "Estimated current at 175°C (A)":find_threshold(175,points),
    })
pd.DataFrame(threshold_rows).round(2)

,Design,Estimated current at 125°C (A),Estimated current at 175°C (A)
0,Baseline 80 mm,32.70,35.53
1,Geometry 1 - 60 mm,31.02,33.73
2,Geometry 2 - 40 mm,28.13,30.64


The 125°C threshold is useful only as a **relative scalability indicator** between the tested designs. The 175°C values must not be interpreted as recommended continuous currents because 175°C is the absolute MOSFET limit.


## 10. Final Recommendation

The final recommendation follows directly from the engineering constraints rather than from temperature ranking alone:

1. **Cooling is necessary:** the 10 A no-heat-sink coupled reference exceeds the device limit.
2. **Aluminium is screened in:** copper's thermal improvement is small relative to its equal-geometry mass/cost penalty.
3. **Geometry is optimised within aluminium:** 40 mm fails the 125°C / 15 W requirement; 80 mm passes but is heavier than necessary; **60 mm is the minimum-mass tested feasible design**.
4. **Scalability is bounded:** the 60 mm design has an estimated 125°C thermal capacity of about 17.3 W under the baseline modelling assumptions.
5. **The recommendation is conditional:** weak natural convection can remove the available thermal margin.

> **Recommended design: aluminium 6061-T6, 60 mm / 6-fin heat sink, fixed TGP5000 TIM.**

### Limitations / pending finalisation

- The analytical-versus-LTspice loss discrepancy still requires a deeper switching-loss explanation.
- Quantitative ANSYS mesh-independence and energy-balance evidence will be added in the final validation pass.
- The full ANSYS project/archive will be added by the materials collaborator.
- The dashboard will be refreshed after the final data freeze.
- TIM optimisation, hybrid heat sinks, CFD/fans and continuous geometry optimisation are intentionally outside scope.
